# 第3章 基礎的なレイトレーシング

書籍『Sionna RTによる電波伝搬レイトレーシング入門』第3章のコードである。
セルは本文の掲載順に並べてあり、コードセルの中身は本文に印刷したものと同じである。

先頭の「準備」セルだけは本文に印刷していない。本文のコードが前提にしている
import とシーンの読み込みをここで済ませてある。準備セルを実行すれば、以降は
上から順に実行できる。

ただし、本文が説明のために示している断片（自分で作ったシーンのパスや、その
シーンには無い物体名を使う例）はそのままでは動かない。該当するセルの前に注記を
置いてある。

## 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

## 1. 最初のシーンを読み込む

In [ ]:
from sionna.rt import load_scene, scene as scene_assets

scene = load_scene(scene_assets.simple_street_canyon)
print(f'objects: {len(scene.objects)}')

In [ ]:
scene.frequency = 3.5e9   # 3.5 GHz (Sub-6 GHz の代表)

## 2. 送信機を配置する

In [ ]:
from sionna.rt import Transmitter, Receiver

tx = Transmitter(name='tx0',
                 position=[-70.0, -6.0, 10.0])  # 単位: メートル
scene.add(tx)

## 3. 受信機を配置する

In [ ]:
rx = Receiver(name='rx0',
              position=[70.0, 6.0, 1.5])
scene.add(rx)

## 4. アンテナを設定する

In [ ]:
from sionna.rt import PlanarArray

# Tx 側: 単素子・垂直偏波・等方性アンテナ
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern='iso',
                             polarization='V')

# Rx 側も同じ設定
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern='iso',
                             polarization='V')

## 5. PathSolver で伝搬パスを計算する

In [ ]:
from sionna.rt import PathSolver

solver = PathSolver()
paths = solver(scene, max_depth=3,
               los=True, specular_reflection=True,
               diffraction=False, diffuse_reflection=False)

# 形状: (rx, tx, パス数)
print('valid:', paths.valid.shape)

## 6. LoS パスを確認する

In [ ]:
import numpy as np

# max_depth=0 で LoS のみ計算
paths_los = solver(scene, max_depth=0, los=True,
                   specular_reflection=False)
print('LoS paths:', int(np.array(paths_los.valid).sum()))
a_los = paths_los.a   # (real, imag) タプル
print('|a| (LoS):', np.sqrt(a_los[0]**2 + a_los[1]**2))

## 7. 反射パスを確認する

In [ ]:
import numpy as np

paths_1 = solver(scene, max_depth=1, los=True,
                 specular_reflection=True)
paths_3 = solver(scene, max_depth=3, los=True,
                 specular_reflection=True)

print('max_depth=1 paths:', int(np.array(paths_1.valid).sum()))
print('max_depth=3 paths:', int(np.array(paths_3.valid).sum()))

## 9. パスを表示する

In [ ]:
v = np.array(paths.vertices)         # 相互作用点の座標
kind = np.array(paths.interactions)  # 相互作用の種類 (0 なら無し)

In [ ]:
from sionna.rt import Camera
cam = Camera(position=[-180, -180, 190], look_at=[0, 0, 0])
fig = scene.render(camera=cam, paths=paths, clip_at=13.0)

## 10. 受信電力と距離の関係を見る

In [ ]:
import numpy as np
# この実験だけ Tx を道の端に置き直す
tx_sweep = np.array([-90.0, -6.0, 10.0])
for dx in np.linspace(20, 160, 22):  # x 方向の離隔 [m]
    scene.remove('rx0')
    rx_pos = np.array([-90.0 + dx, 6.0, 1.5])
    distance = np.linalg.norm(rx_pos - tx_sweep)  # 3 次元距離
    scene.add(Receiver('rx0', position=rx_pos.tolist()))
    a = solver(scene, max_depth=3).a   # (real, imag) タプル
    # パスゲインの合計に 30 dBm を足す
    prx_dbm = 10*np.log10((a[0]**2 + a[1]**2).sum()) + 30

In [ ]:
for depth in [0, 1, 2, 3, 5]:
    paths = solver(scene, max_depth=depth)
    a = paths.a   # (real, imag) タプル
    prx_dbm = 10*np.log10((a[0]**2 + a[1]**2).sum().item()) + 30
    print(f'max_depth={depth}: P_rx = {prx_dbm:.2f} dBm')